# Class 7. Exam prep

## Plan:

<h2>

1) SQL practice <br>

</h2>

## Imports

In [112]:
import pandas as pd
import psycopg2
import os

from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT # to create databases
from pandas.testing import assert_frame_equal

from typing import Iterable, Callable

## Global variables

In [2]:
db_nm = 'postgres'
usr_nm = 'postgres'
host = 'localhost'
port = '5432'

#IMPORTANT! DO NOT ENTER THE PASSWORD HERE, SAVE IT IN THE FILE AND READ IT FROM THE FILE

path_to_pwd_file = 'pwd.txt'

with open(path_to_pwd_file, 'r') as file:
    pwd = file.read().strip('\n')

## SQL practice

### Exercise 1

Connect to the default database, create a new one called `practice`. Use the global variables defined above to test it, the function should account for the cases when the database already exists and NOT raise an error, but print a warning message. You can set the preffered values for the variables as default **except for the password argument**. Create a function to delete a database by name

In [3]:
def connect(pwd: str,
            usr_nm: str = 'postgres', 
            host: str = 'localhost',
            port: str = '5432',
            db_nm: str = 'postgres'
           ) -> psycopg2.extensions.connection | None:

    '''
    Connect to a db

    Inputs:
        usr_nm: str - username for the database
        host: str - host of the sql cluster/ DB
        port: str - target port
        db_nm: str - name of the database to connect to
        pwd: str - user password

    Outputs:
        psycopg2.extensions.connection | None - connection to the database
    '''

    conn = None
    
    try:
        conn = psycopg2.connect(
            host=host,
            database=db_nm,
            user=usr_nm,
            port=port,
            password=pwd
        )

    except:
        print('Connection failure!')

    else:
        print('Connection success!')

    finally:
        return conn


def create_db(new_db_nm: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None
             ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:

    '''
    Create a new db. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection (maybe with cursor)
        is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
        conn: psycopg2.extensions.connection - connection to a database
        cur: psycopg2.extensions.cursor - cursor in a database
        new_db_nm: str - the name of the database to be created

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"

    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

        cur = conn.cursor()

    try:
        cur.execute(f'CREATE DATABASE {new_db_nm};')
    
    except:
        print(f"Couldn't create database. The database {new_db_nm} probably already exists")

    else:
        print(f'Database {new_db_nm} created!')

    finally:
        if conn_flg:
            return conn, cur
        else:
            return cur

def del_db(db_to_del_nm: str,
           conn: psycopg2.extensions.connection | None = None,
           cur: psycopg2.extensions.cursor | None = None
          ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:

    '''
    Delete a target database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection
        (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       db_to_del_nm: str - the name of the database to be created

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''
    
    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

        cur = conn.cursor()

    try:
        cur.execute(f'DROP DATABASE {db_to_del_nm};')
    
    except:
        print(f"Couldn't drop database. The database {db_to_del_nm} probably already exists")

    else:
        print(f'Database {db_to_del_nm} dropped!')

    finally:
        if conn_flg:
            return conn, cur
        else:
            return cur

    

conn = connect(pwd=pwd)
conn, cur = create_db(new_db_nm='practice', conn=conn)

Connection success!
Couldn't create database. The database practice probably already exists


### Exercise 2

Given the file `app_store_reviews_300k_35_columns.csv` create a table with the same name in your new `practice` database. Load the contents of the .csv file in chunks using the `chunksize` argument in `pd.read_csv`. If such a table already exists print out a warning

*Hint: the `chunksize` argument lets you use the result of the `pd.read_csv` function as an Iterable where on each iteration you get a `pd.DataFrame` object*

In [25]:
def create_table(table_nm: str,
                 table_cols: list[str],
                 table_dtypes: list[str],
                 conn: psycopg2.extensions.connection | None = None,
                 cur: psycopg2.extensions.cursor | None = None 
                ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:
    '''
    Create table in a database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection
        (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       table_nm: str - the name of the table to be created

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        cur = conn.cursor()

    
    pairs = []

    for col, dtype in zip(table_cols, table_dtypes):
        pairs.append(' '.join([col, dtype])+',')

    pairs[-1] = pairs[-1].strip(',')

    cols = '\n'.join(pairs)

    # print(cols)
    
    try:
        query = f'''CREATE TABLE {table_nm} (
                id SERIAL,
                {cols}
                    );'''
        print(query)
        
        cur.execute(query)

        conn.commit()

    except Exception as e:
        print('Failed to create table, such table already exists')
        print(str(e))

    else:
        print(f'Succesfully created table {table_nm}')

    finally:

        if conn_flg:
                return conn, cur
        else:
            return cur

def del_table(table_nm: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None
             ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:
    '''
    Create table in a database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection
        (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       table_nm: str - the name of the table to be deleted

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        cur = conn.cursor()

    try:
        cur.execute(f'''DROP TABLE {table_nm};''')
        conn.commit()

    except Exception as e:
        print('Failed to drop table, it probably does not exist')
        # print(str(e))

    else:
        print(f'Succesfully dropped table {table_nm}')

    finally:

        if conn_flg:
                return conn, cur
        else:
            return cur

def upload_to_table(table_nm: str,
                    df: pd.DataFrame,
                    conn: psycopg2.extensions.connection | None = None,
                    cur: psycopg2.extensions.cursor | None = None
                   ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:
    '''
    Upload the pd.DataFrame into the table in the database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur].
        If only connection (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor
        object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       table_nm: str - the name of the table to be deleted
       df: pd.DataFrame - the table to upload

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''
    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        cur = conn.cursor()

    try:
    
        for idx, row_data in df.iterrows():
            cur.execute(f'INSERT INTO {table_nm} ({', '.join(df.columns)}) VALUES ({('%s,'*len(df.columns)).strip(',')})', tuple(row_data))

    except Exception as e:
        print('Failed to append data to SQL table!')
        print(str(e))
        
    else:
        print(f'Added data to table {table_nm}')
    
    finally:
        if conn_flg:
                return conn, cur
        else:
            return cur


# with connect(pwd=pwd) as conn:
    
#     df_tmp = pd.read_csv('app_store_reviews_300k_35_columns.csv', nrows=1)
#     # print(df_tmp)
#     cols = df_tmp.columns
#     dtypes =  [str(dtype) for dtype in df_tmp.dtypes.to_list()]

#     mapping = {'float64' : 'REAL',
#                'int64' : 'INTEGER',
#                'object' : 'VARCHAR(100)'}

#     dtypes = [mapping[dtype] for dtype in dtypes]
    
#     # print(cols, dtypes)
    
#     conn, cur = create_table('app_store', table_cols = cols, table_dtypes=dtypes, conn=conn)
    
#     for idx, df in enumerate(pd.read_csv('app_store_reviews_300k_35_columns.csv', chunksize=1_000)):
#         conn, cur = upload_to_table('app_store', df, conn)

#     conn, cur = del_table('app_store', conn)

In [22]:
with connect(pwd=pwd) as conn:
    create_table('test', ['col_1', 'col_2'], ['REAL', 'VARCHAR(10)'], conn=conn)
    del_table('test', conn=conn)

Connection success!
Succesfully created table test
Succesfully dropped table test


### Exercise 3

Table: `Stadium`

| Column Name   | Type    |
|---------------|---------|
| id            | int     |
| visit_date    | date    |
| people        | int     |

visit_date is the column with unique values for this table.\
Each row of this table contains the visit date and visit id to the stadium with the number of people during the visit.\
As the id increases, the date increases as well.
 

Write a solution to display the records with three or more rows with consecutive id's, and the number of people is greater than or equal to 100 for each.

Return the result table ordered by visit_date in ascending order.

The result format is in the following example.

 

Example 1:

Input:\
Stadium table:

| id   | visit_date | people    |
|------|------------|-----------|
| 1    | 2017-01-01 | 10        |
| 2    | 2017-01-02 | 109       |
| 3    | 2017-01-03 | 150       |
| 4    | 2017-01-04 | 99        |
| 5    | 2017-01-05 | 145       |
| 6    | 2017-01-06 | 1455      |
| 7    | 2017-01-07 | 199       |
| 8    | 2017-01-09 | 188       |

Output: 

| id   | visit_date | people    |
|------|------------|-----------|
| 5    | 2017-01-05 | 145       |
| 6    | 2017-01-06 | 1455      |
| 7    | 2017-01-07 | 199       |
| 8    | 2017-01-09 | 188       |

Explanation: \
The four rows with ids 5, 6, 7, and 8 have consecutive ids and each of them has >= 100 people attended. Note that row 8 was included even though the visit_date was not the next day after row 7.\
The rows with ids 2 and 3 are not included because we need at least three consecutive ids.


For the exercise above you need to upload the data into the SQL table of the same name, then using a SQL query select the neccesary entries. First solve it in python (*Hint: you can use the [`filter`](https://www.w3schools.com/python/ref_func_filter.asp) function for that*) then try yourself in the SQL query (*Hint: keywords [`LAG`](https://www.geeksforgeeks.org/sql/sql-server-lag-function-overview/) and [`LEAD`](https://www.geeksforgeeks.org/postgresql/postgresql-lead-function/) could be of great help*)

In [118]:
def solve_stadium(stadium: pd.DataFrame) -> pd.DataFrame:

    '''
    #python solution

    stadium = stadium[stadium['people'] >= 100]

    # print(stadium)
    def filtering_func(x: int) -> bool:
        arr = stadium['id'].to_list()
        if (x-1 in arr and x+1 in arr) or (x-2 in arr and x-1 in arr) or (x+1 in arr and x+2 in arr):

            # print(f'{x} exists')
            # print(stadium)
            return True
        
        else:
            
            return False
        
    ids = filter(filtering_func, stadium['id'].to_list())

    return stadium[stadium['id'].isin(ids)].sort_values('visit_date')
    '''

    mapping = {
        'int64' : 'BIGINT',
        'object' : 'DATE',
        'datetime' : 'DATE'
    }

    # print(stadium.dtypes.iloc[1:])
    postgre_dtypes = stadium.dtypes.reset_index(drop=False).iloc[1:, :].astype(str)
    postgre_dtypes['new_dtypes'] = postgre_dtypes.iloc[:, 1].apply(lambda x: mapping[x]).to_list()
    # print(postgre_dtypes)

    with connect(pwd=pwd) as conn:
    
        conn, cur = create_table(table_nm='stadium',
                                 table_cols=postgre_dtypes.iloc[:, 0],
                                 table_dtypes=postgre_dtypes.iloc[:, -1],
                                 conn=conn)
        data = None
        
        try:
            conn, cur = upload_to_table(table_nm='stadium',
                                        df=stadium,
                                        conn=conn,
                                        cur=cur)
            
            cur.execute('''
                SELECT id, visit_date, people
                FROM (
                    SELECT  id, visit_date, people,
                            LAG(id, 1, 0) OVER (ORDER BY id) AS id_lag_1,
                            LAG(id, 2, 0) OVER (ORDER BY id) AS id_lag_2,
                            LEAD(id, 1, 0) OVER (ORDER BY id) AS id_lead_1,
                            LEAD(id, 2, 0) OVER (ORDER BY id) AS id_lead_2
                        FROM (
                            SELECT * FROM stadium WHERE people >= 100
                            ) 
                     ) WHERE (id = id_lag_1 + 1 AND id = id_lead_1 - 1) OR
                             (id = id_lag_1 + 1 AND id = id_lag_2 + 2) OR
                             (id = id_lead_1 - 1 AND id = id_lead_2 - 2) ORDER BY visit_date
                ''')
            data = cur.fetchall()

        except Exception as e:
            print('Got an error')
            print(str(e))
            conn.rollback()

        else:
            print('Succesfully executed')
            

        
        finally:
            
            conn, cur = del_table(table_nm='stadium', conn=conn, cur=cur)
            df = pd.DataFrame(columns=['id', 'visit_date', 'people'], data=data)
            df['id'] = df['id'].astype('int64')
            df['people'] = df['people'].astype('int64')
            return df



        # print(stadium.head(3))
        # print(stadium.dtypes)

        

ins, outs = [], []

ins_path = os.path.join('tests', 'Exercise_3', 'inputs')
outs_path = os.path.join('tests', 'Exercise_3', 'outputs')


for file_in_path, file_out_path in zip(filter(lambda x: x.endswith('.csv'), os.listdir(ins_path)),
                                       filter(lambda x: x.endswith('.csv'), os.listdir(outs_path))
                                      ):
    ins.append(pd.read_csv(os.path.join(ins_path, file_in_path), index_col=0))
    outs.append(pd.read_csv(os.path.join(outs_path, file_out_path), index_col=0))

total = 0
for idx, (_in, out) in enumerate(zip(ins, outs)):
    got = solve_stadium(_in)
    if assert_frame_equal(out.reset_index(drop=True), got.reset_index(drop=True), check_dtype=False):
        total += 1
    else:
        # print(got.astype(out.dtypes).dtypes)
        # print(out.dtypes)
        print(f'Test {idx} failed:\nInput:\n{_in}\n\nExpected:\n{out.reset_index(drop=True)}\n\nGot:\n{got.reset_index(drop=True)}')

print(f'You passed {total} / 10 tests!')

Connection success!
CREATE TABLE stadium (
                id SERIAL,
                visit_date DATE,
people BIGINT
                    );
Succesfully created table stadium
Added data to table stadium
Succesfully executed
Succesfully dropped table stadium


AssertionError: DataFrame.iloc[:, 1] (column name="visit_date") are different

DataFrame.iloc[:, 1] (column name="visit_date") values are different (100.0 %)
[index]: [0, 1, 2]
[left]:  [2010-01-03, 2010-01-04, 2010-01-05]
[right]: [2010-01-03, 2010-01-04, 2010-01-05]
At positional index 0, first diff: 2010-01-03 != 2010-01-03

## What to do after?

Keep solving the previous exercises from class 4 in ***fully in SQL***

In [ ]:
###your attempts here